# SAFE ML Txns Endpoint

This notebook builds a clustering model (K-Means) and publishes it as an Amazon SageMaker endpoint, as well as testing it with data in S3 and saving the predictions. It is designed for batch/near-real-time flows within the blossom-analytics-safe-dev-nv bucket.

In [14]:
import boto3
import os

### (Don't run) Step 1: Prepare the file **model.tar.gz**

Download the files needed for inference from S3 and packaging for SageMaker by create model.tar.gz with all artifacts 

* preprocessing_pipeline.joblib (preprocessing pipeline)
* selected_features.csv (features used)
* kmeans_model.joblib (model)
* centroids.csv (clustering centroids)
* kmeans_artifacts.json (metadata)

In [7]:
import tarfile

files_to_download = {
    "kmeans_model.joblib": "output/kmeans/kmeans_analysis/artifact/kmeans_model.joblib",  #artifacts/kmeans_model.joblib
    "preprocessing_pipeline.joblib": "output/preprocessing/preprocessing_pipeline.joblib",  
    "selected_features.csv": "output/feature_selection/selected_features.csv",
    "centroids.csv": "output/kmeans/centroids/centroids.csv",
    "kmeans_artifacts.json": "output/kmeans/kmeans_analysis/artifact/kmeans_artifacts.json"
}

bucket_name = 'blossom-analytics-safe-dev-nv' ## definir con Ramos
local_dir = "endpoint/model_files" ## a definir carpetas locales
os.makedirs(local_dir, exist_ok=True)

s3 = boto3.client("s3")

# Download each file
for local_name, s3_key in files_to_download.items():
    local_path = os.path.join(local_dir, local_name)
    print(f"Downloading s3://{bucket_name}/{s3_key} → {local_path}")
    s3.download_file(bucket_name, s3_key, local_path)

# Create model.tar.gz
tar_path = "model.tar.gz"
with tarfile.open(tar_path, "w:gz") as tar:
    for filename in files_to_download.keys():
        tar.add(os.path.join(local_dir, filename), arcname=filename)

print(f"File generated: {tar_path}")

File generated: model.tar.gz


### (Don't run) Step 2: Upload model.tar.gz to S3

Upload model.tar.gz with all artifacts to S3 at s3://blossom-analytics-safe-dev-nv/output/

In [66]:
model_key = "output/kmeans-endpoint-last3/model.tar.gz"  # a definir nueva ruta en S3
s3.upload_file("model.tar.gz", bucket_name, model_key)
model_artifact_uri = f"s3://{bucket_name}/{model_key}"
print("S3 URI for SKLearnModel:", model_artifact_uri)

S3 URI for SKLearnModel: s3://blossom-analytics-safe-dev-nv/output/kmeans-endpoint-last3/model.tar.gz


### Step 3: Deploy the endpoint with SKLearnModel

V2: Previous Endpoint
* **model_artifact_uri** = "s3://blossom-analytics-safe-dev-nv/output/kmeans-endpoint-final/model.tar.gz"
* **entry_point** = "new-inference.py"
* **endpoint_name** = "final-safe-txns-endpoint"

V3: Final Endpoint 
* **model_artifact_uri** = "s3://blossom-analytics-safe-dev-nv/output/kmeans-endpoint-last3/model.tar.gz"
* **entry_point** = "new-inference-last.py"
* **endpoint_name** = "data-safe-txns-endpoint"

In [ ]:
from sagemaker.sklearn.model import SKLearnModel
from sagemaker import get_execution_role, Session
import sagemaker

# Setup
sagemaker_session = sagemaker.Session()
role = get_execution_role()

# S3 path
#!!!!
#V2
# model_artifact_uri = "s3://blossom-analytics-safe-dev-nv/output/kmeans-endpoint-final/model.tar.gz"  # Previous endpoint (V2)
#V3
model_artifact_uri = "s3://blossom-analytics-safe-dev-nv/output/kmeans-endpoint-last3/model.tar.gz" # Final endpoint (V3)

# Create the model
sk_model = SKLearnModel(
    model_data=model_artifact_uri,
    role=role,
    entry_point="inference_rules.py",
    source_dir="endpoint",
    framework_version="1.2-1",
    sagemaker_session=sagemaker_session
)

In [7]:
# Deploy the endpoint
predictor = sk_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    #!!!!
    endpoint_name="data-safe-txns-endpoint"
    #V2: final-safe-txns-endpoint
    #V3: data-safe-txns-endpoint
)

------!

### Step 4: Test the endpoint from your local machine

In [2]:
import boto3
import pandas as pd
import io
import json

In [16]:
# Parameters
#change with the name of your input file and desired output file
input_key = "real_time/real-time-col-amucu.csv" #new-real-time-example-rename.csv 
output_key = "real_time/output_endpoint/result-real-time-col-amucu-rules.csv"

In [3]:
input_key = "real_time/data_collection_fraudwp.csv"
output_key = "real_time/output_endpoint/result_data_collection_fraudwp.csv.csv"

In [4]:
%%time
#!!!!
endpoint_name = "data-safe-txns-endpoint" 
#V2: final-safe-txns-endpoint
#V3: data-safe-txns-endpoint

region = "us-east-1"
bucket_name = 'blossom-analytics-safe-dev-nv'

# Download CSV from S3
s3 = boto3.client("s3")
response = s3.get_object(Bucket=bucket_name, Key=input_key)
df = pd.read_csv(response["Body"])
print(f"File loaded with {len(df)} rows")

# Convert to CSV in memory (header=True, index=False)
csv_buffer = io.StringIO()
df.to_csv(csv_buffer, header=True, index=False)
payload = csv_buffer.getvalue()

# Invoke the endpoint
runtime = boto3.client("sagemaker-runtime", region_name=region)
response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="text/csv",
    Body=payload
)

# Read and parse results
result = response['Body'].read().decode('utf-8')
parsed = json.loads(result)

File loaded with 12 rows
CPU times: user 91.4 ms, sys: 48.7 ms, total: 140 ms
Wall time: 495 ms


In [5]:
# Convert to DataFrame
df_result = pd.DataFrame(parsed)

# Save in S3
output_buffer = io.StringIO()
df_result.to_csv(output_buffer, index=False)
s3.put_object(Bucket=bucket_name, Key=output_key, Body=output_buffer.getvalue())

print(f"Result saved in s3://{bucket_name}/{output_key}")

Result saved in s3://blossom-analytics-safe-dev-nv/real_time/output_endpoint/result_data_collection_fraudwp.csv.csv


### ! Delete endpoint

In [5]:
client = boto3.client("sagemaker")
endpoint_config_name = "data-safe-txns-endpoint" #data-safe-txns-endpoint #"final-safe-txns-endpoint"

try:
    client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
    print(f"EndpointConfig '{endpoint_config_name}' deleted.")
except client.exceptions.ClientError as e:
    if "Could not find endpoint configuration" in str(e):
        print("EndpointConfig does not exist, continuing.")
    else:
        raise

EndpointConfig 'data-safe-txns-endpoint' deleted.


In [17]:
# ! Force delete
client = boto3.client("sagemaker", region_name="us-east-1")
client.delete_endpoint(EndpointName=endpoint_config_name)
#client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)

{'ResponseMetadata': {'RequestId': 'c637cefc-5623-4bfd-81fe-4c40c6017ef4',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'c637cefc-5623-4bfd-81fe-4c40c6017ef4',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'date': 'Wed, 03 Dec 2025 22:04:13 GMT',
   'content-length': '0'},
  'RetryAttempts': 0}}

#### Additional: Check Endpoint status

In [79]:
import time, boto3
sm = boto3.client("sagemaker")

EP = "data-safe-txns-endpoint"   # o el -v2 si lo cambiaste

for _ in range(120):
    d = sm.describe_endpoint(EndpointName=EP)
    st = d["EndpointStatus"]
    print("status:", st)
    if "FailureReason" in d and d["FailureReason"]:
        print("FailureReason:", d["FailureReason"])   # clave: aquí suele venir el stacktrace del contenedor
        break
    if st in ("InService", "Failed"):
        break
    time.sleep(10)

status: InService
